In [ ]:
import subprocess
import sys
from pathlib import Path

import torch

CODE = next(
    path.parent.parent
    for path in Path("/kaggle/input").rglob("harness.py")
    if path.parent.name == "octlm"
)
OUT = Path("/kaggle/working/day7")
OUT.mkdir(parents=True, exist_ok=True)
MODEL = OUT / "qwen"
if not torch.cuda.is_available() or torch.cuda.get_device_capability(0) != (7, 5):
    raise RuntimeError("Select a Kaggle T4 GPU")
print(
    "Code:", CODE, "PyTorch:", torch.__version__, "GPU:", torch.cuda.get_device_name(0), flush=True
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "tokenizers==0.22.2", "jinja2>=3.1.6,<4"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "octlm.day6", "prepare", "--model", str(MODEL)], cwd=CODE, check=True
)

In [ ]:
def harness(stage, *arguments):
    subprocess.run(
        [
            sys.executable,
            "-m",
            "octlm.harness",
            stage,
            "--out",
            str(OUT / f"{stage}.jsonl"),
            *arguments,
        ],
        cwd=CODE,
        check=True,
    )


harness("validate")
harness("eval", "--model", str(MODEL), "--device", "cuda:0")
print((OUT / "eval.jsonl").read_text().splitlines()[-1], flush=True)